**Бизнес-задача:** Оптимизировать расчет ключевых метрик выручки и социально-демографических показателей клиентов за счет автоматической фильтрации аномалий (выбросов). Это необходимо для точного планирования юнит-экономики и таргетирования маркетинговых кампаний.

Данные по месячным платежам клиентов фитнес-центра, данные занесены в список.

 **Часть 1. Финансовый анализ:** Расчет очищенного среднего платежа пользователя (ARPPU) с контролируемым порогом фильтрации.

In [8]:
import numpy as np

In [14]:
#данные
pay_lst = [
    1600, 800, 3200, 800, 1800, 6400, 1600, 9600, 800, 1800,
    14400, 1600, 7200, 800, 800, 12800, 1800, 45000, 1600, 800, 1800,
    800, 8000, 1600, 9600, 14400, 800, 1600, 800, 1800, 3200,
    800, 1600, 800, 8000, 1800, 40000
]

Средний платеж пользователя

In [15]:
avg_pay = sum(pay_lst)/len(pay_lst)
print("Средний платеж 1 пользователя", round(avg_pay, 2))

Средний платеж 1 пользователя 5745.95


Сортировка списока pay_lst по возрастанию для определения больших значений правого хвоста. Исключение их из расчета среднего

In [17]:
print(sorted(pay_lst))

[800, 800, 800, 800, 800, 800, 800, 800, 800, 800, 800, 1600, 1600, 1600, 1600, 1600, 1600, 1600, 1800, 1800, 1800, 1800, 1800, 1800, 3200, 3200, 6400, 7200, 8000, 8000, 9600, 9600, 12800, 14400, 14400, 40000, 45000]


In [19]:
sum_pay = 0
cnt_pay = 0
for i in pay_lst:
    if i<15000:
        sum_pay +=i
        cnt_pay +=1
avg_pay = sum_pay/cnt_pay
print(f"Новое среднее значение {round(avg_pay, 2)}")

Новое среднее значение 3645.71


Преобразование в функцию

In [23]:
def calc_avg(lst, thresh):
    sum_pay = 0
    cnt_pay = 0
    for i in lst:
        try:
            if i<thresh:
                sum_pay +=i
                cnt_pay +=1
        except TypeError:
            print(f"{i} это нечисловое значение")
    try:
        avg_pay = sum_pay/cnt_pay
        return round(avg_pay, 2)
    except:
        print(f'Список должен содержать хотя бы одно значение меньше {thresh}')


In [24]:
calc_avg(pay_lst, 15000)

3645.71

Нахождение значение thresh для любого списка с помощью кода, чтобы выбросы определялись на основе значений pay_lst

In [27]:
np.mean(pay_lst)

5745.945945945946

In [28]:
np.median(pay_lst)

1800.0

In [29]:
q1 = np.percentile(pay_lst, 25)
q2 = np.percentile(pay_lst, 50)
q3 = np.percentile(pay_lst, 75)
p95 = np.percentile(pay_lst, 95)
print(q1, q2, q3, p95)

800.0 1800.0 7200.0 19519.99999999989


In [30]:
iqr = q3-q1
lower_bound = q1-1.5*iqr
upper_bound = q3+1.5*iqr
print(lower_bound)
print(upper_bound)

-8800.0
16800.0


In [31]:
sum_pay = 0
cnt_pay = 0
for i in pay_lst:
    if (i<upper_bound) and (i>lower_bound):
        sum_pay +=i
        cnt_pay +=1
avg_pay = sum_pay/cnt_pay
print(f"Новое средее значение {round(avg_pay, 2)}")

Новое средее значение 3645.71


<image src="/IQR_range.jpg" alt="Описание изображения">

Функция, которая принимает на вход только список и возвращает среднее значение рассчитанное после исключения выбросов

In [32]:
def calc_avg_adv(lst):
    try:
        q1 = np.percentile(lst, 25)
        q3 = np.percentile(lst, 75)
        iqr = q3-q1
        lower_bound = q1-1.5*iqr
        upper_bound = q3+1.5*iqr
        sum_pay = 0
        cnt_pay = 0
        for i in lst:
            if (i<upper_bound) and (i>lower_bound):
                sum_pay +=i
                cnt_pay +=1
        avg_pay = sum_pay/cnt_pay
        return round(avg_pay, 2)
    except:
        print('Список не должен содержать нечиcловых значений или быть пустым')

In [33]:
calc_avg_adv(pay_lst)

3645.71

In [34]:
lst1 = []
calc_avg_adv(lst1)

Список не должен содержать нечиcловых значений или быть пустым


 **Часть 2. Социально-демографический анализ:** Автоматическое определение границ целевой аудитории по возрасту с использованием метода межквартильного размаха (IQR).

In [2]:
# Рассчет среднего возраста по всему списку age_lst

age_lst = [23, 45, 18, 80, 22, 25, 34, 37, 41, 88, 19, 20, 18, 45, 41, 39, 31, 32, 22, 28, 37]
avg_age=sum(age_lst)/len(age_lst)
print(round(avg_age))

35


In [3]:
#Рассчет среднего возраста для посетителей, которые моложе 50 лет

sum_age=0
cnt=0
for i in age_lst:
    if i<50:
     sum_age+=i
     cnt+=1
avg_age= sum_age/cnt
print(round(avg_age))

30


In [4]:
#Создание функции, которая на вход принимает список и пороговое значение

def avg_age(age_lst, thresh):
    sum_age=0
    cnt=0
    for i in age_lst:
        if i<thresh:
         sum_age+=i
         cnt+=1
    avg_age= sum_age/cnt
    return round(avg_age)

In [5]:
avg_age(age_lst, thresh=50)

30

In [9]:
#Рассчет среднего значения для всех элементов списка, за исключением тех, которые лежат вне интервала [Q1-1,5IQR; Q3+1,5IQR]

q1=np.percentile(age_lst, 25)
q2=np.percentile(age_lst, 50)
q3=np.percentile(age_lst, 75)
q4=np.percentile(age_lst, 95)
print(q1, q2, q3, q4)
iqr=q3-q1
lower_bound=q1-1.5*iqr
upper_bound=q3+1.5*iqr
print(lower_bound, upper_bound)

22.0 32.0 41.0 80.0
-6.5 69.5


In [10]:
sum_age=0
cnt=0
for i in age_lst:
    if (i<upper_bound) and (i>lower_bound):
     sum_age+=i
     cnt+=1
avg_age= sum_age/cnt
print(round(avg_age))

30


In [11]:
# Создание функции, которая принимает на вход список и рассчитывает средний возраст, исключая из расчётов выбросы

def calc_avg_age(age_lst):
    q1=np.percentile(age_lst, 25)
    q3=np.percentile(age_lst, 75)
    iqr=q3-q1
    lower_bound=q1-1.5*iqr
    upper_bound=q3+1.5*iqr
    sum_age=0
    cnt=0
    for i in age_lst:
        if (i<upper_bound) and (i>lower_bound):
         sum_age+=i
         cnt+=1
    avg_age= sum_age/cnt
    return round(avg_age)

In [12]:
calc_avg_age(age_lst)

30